# Practical Work Sheet No. 2 [Machine Learning]
## Ensemble Methods: Bagging, Boosting and Stacking

**Chapter 2 — Supervised Learning**
Dr SERIDI BORDJIBA Y. | November 2026


## 1. Setting Up the Environment

Check the development environment: **Python ≥ 3.10, Jupyter**, and the libraries **numpy, pandas, matplotlib, seaborn, scikit-learn**. In case of reinstallation, from a command prompt:

```bash
pip install numpy pandas matplotlib seaborn scikit-learn jupyter
```

> **Note:** students without a local installation can use Google Colab, which includes all of these libraries.

Run the following cell once, then keep this configuration for the following labs.

In [1]:
# Common imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               AdaBoostClassifier, GradientBoostingClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, roc_auc_score,
                              roc_curve)
sns.set(style='whitegrid')

## 2. Loading and Preparing the Data

The dataset used is the well-known **Breast Cancer Wisconsin (Diagnostic)** dataset, available directly in scikit-learn through `load_breast_cancer()`: no download, no file to place next to the notebook. It contains **569 observations** described by **30 numerical variables** computed from digitized images of aspirates (radius, texture, perimeter, area, smoothness, compactness, concavity, etc.).

| Element | Role | Description |
|---|---|---|
| Variables (30 columns) | Explanatory (X) | 10 imaging measurements (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) × 3 statistics (mean, standard error, worst) |
| `target` | Target (y) | 0 = malignant tumour, 1 = benign tumour |
| Observations | — | 569 (212 malignant, 357 benign) |

In [2]:
# Load the dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# TODO: display the column types (X.info()), the descriptive statistics
# (X.describe()) and the class distribution (y.value_counts())
X.info()
print(X.describe().T)
print(y.value_counts())
...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

Ellipsis

In [4]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (398, 30) Test: (171, 30)


## 3. Decision Trees (CART)

CART decision trees serve here as the **baseline model**: simple and interpretable, they are highly sensitive to overfitting when their depth is not limited. The ensemble methods in the rest of this lab precisely aim at correcting this instability.

**a.** Train a `DecisionTreeClassifier` (default parameters, `random_state=42`) on the training set, then display its accuracy on the test set.

In [5]:
# Train DecisionTreeClassifier (default params)
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
print('Test accuracy (default DT):', clf.score(X_test, y_test))

Test accuracy (default DT): 0.9181286549707602


**b.** Visualize the resulting tree with `plot_tree` (first three levels, feature and class names, `filled=True`).

In [ ]:
# Visualize tree (first 3 levels)
plt.figure(figsize=(12, 6))
plot_tree(clf, max_depth=3, feature_names=data.feature_names,
          class_names=data.target_names, filled=True)
plt.show()

**c. Study the effect of the `max_depth` parameter — TODO**

- Create the list of depths to be tested (for example from 1 to 15).
- For each depth, train a `DecisionTreeClassifier` and record the accuracy on both the training and test sets.
- Plot both accuracy curves as a function of depth and identify the zone where the gap between them widens (train → 1.0 while test stagnates or decreases).

> **Hints:** use two lists `train_acc` and `test_acc`, then `plt.plot()` for the visualization; keep `random_state=42` for reproducibility.

In [ ]:
# TODO: study the effect of max_depth (from 1 to 15)
depths = range(1, 16)
train_acc, test_acc = [], []

for d in depths:
    # TODO: train a DecisionTreeClassifier(max_depth=d, random_state=42)
    # and record its train accuracy and test accuracy
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    test_acc.append( tree.score(X_test, y_test))
plt.figure(figsize=(8, 5))
plt.plot(depths, train_acc, marker='o', label='Train accuracy')
plt.plot(depths, test_acc, marker='o', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision tree: train vs test accuracy by depth')
plt.xticks(depths)
plt.legend()
plt.show()
    


# TODO: plot both accuracy curves as a function of depth (plt.plot)
...

> **Question 1:** which depth minimizes overfitting, and why?

## 4. Random Forest (Bagging)

Bagging (**bootstrap aggregating**) trains many trees on bootstrap samples of the training set, then aggregates their predictions by majority vote: **it reduces the variance without increasing the bias**.

**a.** Train a `RandomForestClassifier` (`n_estimators=100`, `random_state=42`), display its accuracy on the test set and compare it with the single tree.

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('Random Forest test accuracy:', rf.score(X_test, y_test))

**b.** Display the feature importances (`rf.feature_importances_`) sorted in decreasing order (top ten) and comment on the dominant features.

Compare the performance and stability of the forest with those of the single tree of Part 3.

In [ ]:
# Feature importances
importances = pd.Series(rf.feature_importances_,
                        index=data.feature_names)
print(importances.sort_values(ascending=False).head(10))

**TODO — Comments:**

- Which features dominate, and what do they measure physically?
- Compare the accuracy of the forest with the single tree, and their stability.


## 5. Boosting (AdaBoost and Gradient Boosting)

Boosting sequentially builds a series of weak classifiers, each focusing on the examples misclassified by the previous ones: **it essentially reduces the bias**.

**a. Implement AdaBoost from scratch (binary classification) — TODO.** The algorithm is the following:

1. Initialize the sample weights uniformly: wi = 1/n.
2. At each iteration t:
   - train a weak classifier (decision stump: `DecisionTreeClassifier(max_depth=1)`) on the data weighted by w;
   - compute the weighted error of the stump on the training set;
   - compute the coefficient αt = 0.5 × log((1 − error) / error);
   - increase the weight of the misclassified examples, then normalize the weight vector.
3. The final prediction is the weighted vote: H(x) = sign(Σ αt ht(x)).

> **Tips:** weight the training of the stumps with the `sample_weight` parameter of `fit()`; check that each stump does better than chance (weighted error < 0.5); keep `random_state=42`.
>
> **Be careful:** the update rule assumes labels in {−1, +1}, while `target` is coded in {0, 1}. Convert the labels (e.g. 2y − 1) inside `update_weights` and in the final vote.

In [6]:
# TODO: complete the four helper functions
def initialize_weights(n):
    """Uniform weight vector: w_i = 1/n."""
    wi = np.one(n)/ n
    
    return wi

def compute_error(y, y_pred, w):
    """Weighted error: sum of the weights of the misclassified examples."""
    ce=np.sum(w[y != y_pred])
    return ce

    

def compute_alpha(ce):
    """Model weight: alpha = 0.5 * log((1 - error) / error)."""
    a = 0.5*np.log((1-ce)/ce)
    return a
    

def update_weights(y, y_pred, w, alpha):
    """Update the weights (misclassified examples gain weight), then normalize."""
    y = 2 * y - 1
    y_pred = 2 * y_pred - 1
    w *= np.exp(-alpha * y * y_pred)
    return w / np.sum(w)

In [ ]:
# TODO: complete the training and prediction functions
def adaboost_train(X, y, n_estimators=50):
    """Train AdaBoost; returns the list of (stump, alpha) pairs."""
    models = []
    w = initialize_weights(len(y))
    for t in range(n_estimators):
        stump =DecisionTreeClassifier(max_depth=1, random_state=42)
        stump.fit(X, y, sample_weight=w)
        y_pred = stump.predict(X)
        ce = compute_error(y, y_pred,w)
        a = compute_alpha(ce)
        w = update_weights(y, y_pred,w,a)
        models.append((stump, a))
   

    ...

def adaboost_predict(X, models):
    """Final prediction: weighted vote H(x) = sign(sum_t alpha_t * h_t(x))."""
    ...

**b.** Train your implementation with 50 stumps, evaluate the accuracy on the test set, then add the result to the summary table of Part 6.

In [ ]:
# TODO: train your AdaBoost with 50 stumps and evaluate it on the test set
models_adaboost = ...
y_pred_adaboost = ...
acc_adaboost = ...
print('AdaBoost (from scratch) test accuracy:', acc_adaboost)

**c.** Train a `GradientBoostingClassifier` (`n_estimators=100`, `random_state=42`) provided by scikit-learn, evaluate it on the test set and compare it with the previous methods.

*Each tree predicts the residuals of the current model (the gradient of the loss); the update writes Ft(x) = Ft−1(x) + ν ht(x), where ν is the learning rate. AdaBoost can be seen as the special case obtained with the exponential loss.*

In [ ]:
# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
print('Gradient Boosting test accuracy:', gb.score(X_test, y_test))

## 6. Comparison and Visualizations

**a.** Gather the model accuracies in a summary table (pandas), adding your from-scratch AdaBoost and, after Part 7, the stacking.

In [ ]:
# Compare models and build summary table
models = {'DecisionTree': clf, 'RandomForest': rf, 'GradBoost': gb}
results = []
for name, m in models.items():
    y_pred = m.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': acc})

# TODO: add the from-scratch AdaBoost results, then the stacking ones (Part 7)
# Hint: for the AUC column you need a continuous score — m.predict_proba(X_test)[:, 1]
# for the scikit-learn models, adaboost_score(X_test, models_adaboost) for your AdaBoost
...

res_df = pd.DataFrame(results)
res_df

**Table to complete:**

| Model | Accuracy (test) | AUC (test) | Remarks |
|---|---|---|---|
| DecisionTree | | | |
| RandomForest | | | |
| AdaBoost (from scratch) | | | |
| GradientBoosting | | | |
| Stacking | | | |

**b.** Plot the ROC curves of the different models on the same figure (`roc_curve`) and compute the corresponding AUC (`roc_auc_score`).

> **Tip:** the ROC curve needs a **continuous score**, not predicted labels: use `predict_proba(X_test)[:, 1]` for the scikit-learn models, and the weighted sum Σ αt ht(x) for your AdaBoost (before the `sign`).

In [ ]:
# TODO: plot the ROC curves of the models on the same figure
# and compute the corresponding AUC (roc_curve, roc_auc_score)
plt.figure(figsize=(8, 6))
...

**c.** Display the confusion matrix (`confusion_matrix`) and the classification report (`classification_report`) of the best model, then interpret the four cells in the medical context: do the two classes have the same error cost?

In [ ]:
# TODO: display the confusion matrix and the classification report
# of the best model, then interpret the four cells
best_model = ...
y_pred_best = ...
print(confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best, target_names=data.target_names))

## 7. Stacking

Stacking combines several heterogeneous models through a meta-estimator, learned on the (out-of-sample) predictions of the base models.

**Implement a simple stacking — TODO:**

- Define the three base models: `LogisticRegression`, `RandomForestClassifier` and `GradientBoostingClassifier`.
- Define the meta-model (final estimator): `LogisticRegression`.
- Train a `StackingClassifier` with 5-fold cross-validation (`cv=5`).
- Evaluate its accuracy on the test set, compare it with the previous models and complete the summary table.

> **Tips:** use `StackingClassifier` from `sklearn.ensemble`; compare the accuracy with RF, GB and AdaBoost (see Question 6). Think about `max_iter=1000` (or more) for the logistic regressions.

In [ ]:
# TODO: implement the stacking
#  - base models: LogisticRegression, RandomForestClassifier, GradientBoostingClassifier
#  - meta-model: LogisticRegression
#  - StackingClassifier(..., cv=5)
stack = ...
stack.fit(X_train, y_train)
print('Stacking test accuracy:', ...)

In [ ]:
# TODO: add the stacking to the summary table (res_df) and display it
...

## 8. Analysis Questions

Answer concisely, directly in the cells below.

#### Question 1. Which depth (`max_depth`) minimizes overfitting, and why?

#### Question 2. Explain why boosting can reduce the bias while being sensitive to noise (overfitting).

#### Question 3. Interpret the results of the summary table: for which use case (little noise, a lot of noise, need for interpretability) would you recommend each method?

#### Question 4. Why is stratification (`stratify=y`) important when splitting into train/test sets? What could happen to the evaluation if it were omitted?

#### Question 5. Using the bias-variance decomposition, explain why bagging improves the stability of a decision tree. Why is each tree of the forest trained on a bootstrap sample and with a subset of features?

#### Question 6. In stacking, why must the meta-model be learned on predictions obtained through cross-validation rather than on the base models’ predictions on the training set?